# CacaoTrace — Entrenar los modelos de visión

Este cuaderno entrena los modelos que la app usa **dentro del teléfono, sin internet**.
No necesitas instalar nada en tu computadora: todo corre en los servidores de Google.

| Modelo | Qué hace | Dónde se usa en la app |
|---|---|---|
| **M1 mazorca** | sana / monilia / fitóftora / otro | Recepción (RF-REC-02) |
| **M2 prueba de corte** | detecta y clasifica cada grano cortado | Prueba de corte (RF-PRC-03) |
| **M3 tostado** | crudo / ligero / medio / oscuro / quemado | Tostado (RF-TOS-03) |
| **M4 chocolate** | atemperado_ok / fat_bloom / sugar_bloom / sin_brillo | Atemperado (RF-ATE-04) |

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.
Sin GPU el entrenamiento tarda horas en vez de minutos.


## Paso 1. Conectar tu Google Drive y preparar el entorno


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambia esta ruta si guardaste el proyecto en otro lugar de tu Drive
PROYECTO = '/content/drive/MyDrive/CacaoTrace/entrenamiento'
%cd $PROYECTO
!pip install -q -r requirements.txt


In [ ]:
# Comprobar que la GPU está encendida
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPU disponible:', gpus if gpus else 'NO — ve a Entorno de ejecución → Cambiar tipo')


## Paso 2. Comprobar que todo funciona (5 minutos, sin fotos)

Genera imágenes falsas y corre el pipeline completo. Si esto termina en verde,
el entorno está bien y cualquier problema posterior será de tus datos, no del código.


In [ ]:
!bash herramientas/smoke_test.sh


## Paso 3. Colocar tus fotos

Dentro de `fuentes/` crea **una carpeta por origen** y, dentro, **una carpeta por clase**:

```
fuentes/
  fotos_app/                 <- tus fotos exportadas desde CacaoTrace
    sana/  monilia/  fitoftora/  otro/
  CocoaMoniliaDataSet/       <- dataset abierto
    h0/  m1/  m2/  m3/
```

Las fotos que exporta la app ya vienen así desde `Drive/CacaoTrace/Dataset/mazorca/`.

> **Licencias.** Usa solo datasets que permitan uso comercial (por ejemplo CC BY 4.0)
> y tus propias fotos. El dataset de prueba de corte de Santos et al. (2019) es
> CC BY-NC-ND 4.0 y **no** puede usarse para un modelo que vayas a vender.


In [ ]:
# Ver qué hay en fuentes/ y cuántas fotos tiene cada clase
from pathlib import Path
for fuente in sorted(Path('fuentes').iterdir()):
    if not fuente.is_dir():
        continue
    print(f'\n{fuente.name}')
    for clase in sorted(fuente.iterdir()):
        if clase.is_dir():
            n = sum(1 for _ in clase.rglob('*') if _.suffix.lower() in {'.jpg','.jpeg','.png'})
            print(f'   {clase.name:<20} {n:>5} fotos')


## Paso 4. Escribir el mapeo de clases

Cada dataset llama distinto a lo mismo. El mapeo traduce sus nombres a los de la app.
Usa `null` para descartar una clase que no te sirve.


In [ ]:
import json

mapeo = {
    'fotos_app':           {'sana':'sana', 'monilia':'monilia', 'fitoftora':'fitoftora', 'otro':'otro'},
    'CocoaMoniliaDataSet': {'h0':'sana', 'm1':'monilia', 'm2':'monilia', 'm3':'monilia'},
    # 'Cacao_Diseases_Pests': {'HEALTHY':'sana', 'FROSTYPOD':'monilia', 'BLACKPOD':'fitoftora', 'MIRID':'otro'},
}
Path('mapeo_mazorca.json').write_text(json.dumps(mapeo, indent=2, ensure_ascii=False))
print(open('mapeo_mazorca.json').read())


## Paso 5. Preparar el dataset

Une las fuentes, corrige la rotación de las fotos del teléfono, elimina duplicados
y divide en entrenamiento / validación / prueba.

`--agrupar_por carpeta` evita que dos fotos de **la misma mazorca** caigan una en
entrenamiento y otra en prueba: si eso pasa, el modelo parece mucho mejor de lo que es.


In [ ]:
!python preparar_dataset.py \
    --fuentes fuentes \
    --mapeo mapeo_mazorca.json \
    --salida datos/mazorca \
    --lado_max 1024


## Paso 6. Entrenar M1 (mazorca)

Son dos fases automáticas:
1. **Cabeza nueva**: la red ya sabe ver formas y texturas; solo aprende a separar tus clases.
2. **Ajuste fino**: se reentrenan las últimas capas con pasos muy pequeños.

Al final exporta a `.tflite` y comprueba que el archivo responde **lo mismo** que el
modelo original. Si la concordancia baja de 0,98, la conversión rompió algo.


In [ ]:
!python entrenar_clasificador.py \
    --datos datos/mazorca \
    --tarea mazorca \
    --arquitectura efficientnetv2b0 \
    --epocas 15 --epocas_ajuste 15


### Leer el resultado

Lo que importa NO es la exactitud general, sino el **recall de monilia y fitóftora**:
el error caro es decir «sana» a una mazorca enferma.

El script te dice directamente si el modelo cumple las metas para poder usarse.


In [ ]:
from IPython.display import Image, display
import json

print(open('salidas/mazorca/reporte.txt').read())
meta = json.load(open('salidas/mazorca/metadatos.json'))
print('¿Se puede activar en la app?', 'SÍ' if meta['cumple_metas_liberacion'] else 'TODAVÍA NO')
for p in meta.get('problemas', []):
    print('  -', p)
display(Image('salidas/mazorca/matriz_confusion.png'))
display(Image('salidas/mazorca/curvas.png'))


## Paso 7. Comparar con el modelo que ya tiene la app

Nunca publiques un modelo solo porque es más nuevo. Esto los mide a los dos con
**las mismas imágenes** y avisa si el nuevo detecta peor las enfermedades.


In [ ]:
!python comparar_modelos.py \
    --actual ../app/assets/modelos/mazorca/modelo.tflite \
    --nuevo salidas/mazorca/mazorca_int8.tflite \
    --datos datos/mazorca/test \
    --etiquetas salidas/mazorca/etiquetas.txt \
    --criticas monilia fitoftora


## Paso 8. Instalar el modelo en la app


In [ ]:
!python herramientas/instalar_modelo.py --tarea mazorca


---
## M2 — Detector de la prueba de corte

Este es distinto: no clasifica la foto entera, sino que **encuentra cada grano** y lo
clasifica. Necesita fotos etiquetadas caja por caja, hechas con Roboflow o CVAT.

Mínimo práctico: **40 tableros** (unos 4.000 granos). Recomendado: 100 o más.


In [ ]:
!python entrenar_detector_corte.py --datos datos/corte/data.yaml --epocas 100


### Medir lo que de verdad importa

El mAP50 puede verse bien y el conteo seguir mal (por ejemplo, si el modelo parte un
grano en dos cajas). Esto mide el error de conteo y el error en el % de fermentados,
que es lo que decide el grado del lote.


In [ ]:
!python entrenar_detector_corte.py --evaluar_conteo datos/corte/data.yaml


In [ ]:
# Probar con una foto suelta de un tablero
!python entrenar_detector_corte.py --probar mi_tablero.jpg


---
## M3 (tostado) y M4 (chocolate)

Mismo procedimiento que M1, solo cambian las carpetas y el nombre de la tarea.
Estos dos **solo pueden entrenarse con tus propias fotos**: no hay datasets abiertos.


In [ ]:
!python preparar_dataset.py --fuentes fuentes --mapeo mapeo_tostado.json --salida datos/tostado
!python entrenar_clasificador.py --datos datos/tostado --tarea tostado
!python herramientas/instalar_modelo.py --tarea tostado


---
## Recordatorio final

Los modelos son **apoyo, no autoridad**. En la app el usuario siempre puede corregir
el resultado, la corrección manda y se guarda para reentrenar. El grado de calidad y
el cadmio con validez legal dependen de la norma oficial y de un laboratorio
acreditado (RF-IA-07).
